# Gráficos honestos

Notebook da aula **Gráficos honestos** (visualização de dados na prática, quinta,
12/11). O notebook tem uma parte para cada bloco prático da aula, na mesma ordem:

0. os dados e um primeiro gráfico
1. os princípios em código: redesenhe três gráficos que os quebram
2. escolha o gráfico: cinco perguntas, o gráfico é você quem escolhe
3. carros furtados e roubados: contagem e taxa
4. conserte este gráfico: em duplas

Entre as partes 1 e 2 está a caixa de ferramentas dos slides, uma célula por
gráfico, para copiar; depois da parte 2, as duas revisões de "como gráficos mentem"
em código.

**Como rodar.** No Google Colab já vem tudo o que este notebook usa (pandas,
seaborn, matplotlib, scikit-learn): é só rodar as células. No seu computador,
instale antes: `pip install pandas seaborn scikit-learn jinja2` (seaborn 0.13 ou
mais novo; jinja2 formata as tabelas).

**Dados.** A base da aula, `fogo_cruzado_rio.csv`: 27.738 registros de tiroteios
na cidade do Rio de Janeiro, de 5 de julho de 2016 a 24 de novembro de 2025, um
registro por linha, 16 colunas. Fonte: Instituto Fogo Cruzado
(api.fogocruzado.org.br); extração preparada para a aula, campos simplificados. É
só para uso na aula: não republique. A célula de carga usa uma cópia em `data/`
se houver, senão o link passado em aula (`DATA_URL`), senão pede para você
enviar o arquivo. A aula trabalha com o período de 01/01/2023 a 24/11/2025
(`WINDOW_START`); a série completa só aparece onde uma tendência precisa dela. Os
nomes das colunas ficam em inglês, como no arquivo.

Antes de ler qualquer gráfico, lembre o que a tabela não diz: registro não é
evento (um registro é o que chegou ao Fogo Cruzado); não há coluna de população,
então contagem não é risco; `main_reason` é "Não identificado" em 61% dos
registros; 78% não listam vítimas; um registro (Penha, 28/10/2025) tem 145
vítimas; 2016 começa em julho e 2025 termina em 24 de novembro.

As células marcadas **A FAZER** são suas: escreva o código onde a célula diz `# seu código aqui`. A solução fica escondida logo abaixo de cada exercício; "executar tudo" roda as soluções e define o que as células seguintes precisam. Quando uma pergunta tem mais de um bom gráfico, a solução diz.

## Preparação

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, ScalarFormatter


def br(x, decimals=0):
    """Um número no padrão brasileiro: 27.738 e 13,62."""
    text = f"{x:,.{decimals}f}"
    return text.replace(",", "_").replace(".", ",").replace("_", ".")


# Eixos com ponto de milhar e vírgula decimal: 1.000 e 2,5.
BR = FuncFormatter(lambda x, pos: br(x, 0 if float(x).is_integer() else 1))


# Todos os eixos no padrão brasileiro, sem repetir isso em cada gráfico
# (pode rodar esta célula de novo: o original fica guardado uma vez só).
if not hasattr(ScalarFormatter, "_call_original"):
    ScalarFormatter._call_original = ScalarFormatter.__call__
if not hasattr(ScalarFormatter, "_set_locs_original"):
    ScalarFormatter._set_locs_original = ScalarFormatter.set_locs


def _set_locs_br(self, locs):
    self._br_locs = [v for v in locs if v == v]   # para reconhecer um eixo de anos
    ScalarFormatter._set_locs_original(self, locs)


def _scalar_br(self, x, pos=None):
    text = ScalarFormatter._call_original(self, x, pos)
    plain = text.replace("\u2212", "-")
    try:
        value = float(plain)
    except ValueError:
        return text
    decimals = len(plain.split(".")[1]) if "." in plain else 0
    locs = getattr(self, "_br_locs", [])
    if decimals == 0 and locs and min(locs) >= 1900 and max(locs) <= 2100:
        return plain                    # um eixo de anos: 2025, não 2.025
    return br(value, decimals).replace("-", "\u2212")


ScalarFormatter.set_locs = _set_locs_br
ScalarFormatter.__call__ = _scalar_br

## Parte 0: os dados

Carregue o CSV e desenhe o número de registros por ano.

O slide mostra a linha que importa, `pd.read_csv(FC_PATH, parse_dates=["date"])`; a célula abaixo antes garante que o arquivo esteja em `FC_PATH`: uma cópia em `data/`, senão o link da aula em `DATA_URL`, senão o botão de envio do Colab.

In [ ]:
# A base da aula: fogo_cruzado_rio.csv (27.738 registros, 16 colunas).
# Fonte: Instituto Fogo Cruzado (api.fogocruzado.org.br); extração preparada para a aula.
# É um arquivo da aula, não um conjunto aberto: use na aula, não redistribua.
import re
import urllib.request
from pathlib import Path

# >>> PLACEHOLDER: o professor passa este link. Cole entre as aspas. <<<
# Um link comum de compartilhamento do Google Drive funciona ("Qualquer pessoa com o link" tem de estar ligado).
DATA_URL = "PASTE-THE-LINK-TO-fogo_cruzado_rio.csv-HERE"


def direct_link(url):
    """Um link de compartilhamento do Google Drive vira o link de download direto; qualquer outra URL fica como está."""
    m = re.search(r"drive\.google\.com/.*?(?:/d/|[?&]id=)([\w-]+)", url)
    return ("https://drive.usercontent.google.com/download?id=" + m.group(1)
            + "&export=download") if m else url


FC_PATH = Path("data/fogo_cruzado_rio.csv")
FC_PATH.parent.mkdir(parents=True, exist_ok=True)

# 1. Já está aqui (de uma execução anterior, ou você copiou para data/)? Use.
# 2. Senão, baixe de DATA_URL, depois de preenchida.
if not FC_PATH.exists() and direct_link(DATA_URL).startswith("http"):
    try:
        urllib.request.urlretrieve(direct_link(DATA_URL), FC_PATH)
        if FC_PATH.read_bytes()[:1] == b"<":     # uma página da web, não o CSV (o link não é público?)
            FC_PATH.unlink()
            print("DATA_URL devolveu uma página da web, não o arquivo: o link está aberto a qualquer pessoa?")
        else:
            print("baixado", FC_PATH)
    except Exception as error:
        print("não foi possível baixar de DATA_URL:", error)

# 3. Último recurso: escolha o arquivo no seu computador (só no Colab).
if not FC_PATH.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            f"fogo_cruzado_rio.csv não está em {FC_PATH.resolve()}: coloque-o lá, "
            "ou cole o link em DATA_URL acima") from None
    print("Escolha fogo_cruzado_rio.csv no seu computador:")
    uploaded = files.upload()
    Path(next(iter(uploaded))).replace(FC_PATH)

WINDOW_START = "2023-01-01"   # o período da aula: de 01/01/2023 a 24/11/2025

df = pd.read_csv(FC_PATH, parse_dates=["date"])
if df.shape != (27738, 16):
    raise RuntimeError(f"esperava 27.738 registros e 16 colunas, veio {df.shape}: "
                       "este é o CSV da aula?")
recent = df[df["date"] >= WINDOW_START]
print(f"{br(len(df))} registros, de {df['date'].min():%d/%m/%Y} a "
      f"{df['date'].max():%d/%m/%Y}; {br(len(recent))} desde {WINDOW_START}")

Os nomes de bairro vêm em maiúsculas e sem acento (`PRACA SECA`). Para os gráficos, a função `bairro` os escreve como se escreve (`Praça Seca`):

In [ ]:
ACENTOS = {"MARE": "Maré", "PRACA SECA": "Praça Seca", "BRAS DE PINA": "Brás de Pina",
           "ITANHANGA": "Itanhangá", "COMPLEXO DO ALEMAO": "Complexo do Alemão",
           "ANDARAI": "Andaraí", "GAVEA": "Gávea", "IRAJA": "Irajá", "MEIER": "Méier",
           "SAO CONRADO": "São Conrado", "SAO CRISTOVAO": "São Cristóvão",
           "SAO FRANCISCO XAVIER": "São Francisco Xavier",
           "PRACA DA BANDEIRA": "Praça da Bandeira"}


def bairro(nome):
    """PRACA SECA -> Praça Seca: o arquivo traz os nomes em maiúsculas, sem acento."""
    if nome in ACENTOS:
        return ACENTOS[nome]
    return " ".join(p if p in ("da", "de", "do", "das", "dos") else p.capitalize()
                    for p in nome.lower().split())

In [ ]:
df.head()

In [ ]:
df.describe()

**A FAZER 0.1** Desenhe o primeiro gráfico: registros por ano, em barras.

In [ ]:
# reports_year = ____  # registros por ano, em ordem de ano
# ax = reports_year.plot.bar(rot=0)
# ax.yaxis.set_major_formatter(BR)
# ax.set(xlabel="", ylabel="registros")
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
reports_year = df["year"].value_counts().sort_index()
ax = reports_year.plot.bar(rot=0)
ax.yaxis.set_major_formatter(BR)
ax.set(xlabel="", ylabel="registros")
plt.show()

**0.2** Por que 2016 e 2025 são tão baixos, e o que conta um registro?

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# **0.2** Por que 2016 e 2025 são tão baixos, e o que conta um registro?
#
# *Resposta:* os dois são anos parciais: o Fogo Cruzado começa no Rio em 5 de julho de 2016 e o arquivo termina em 24 de novembro de 2025. Um registro é o que chegou ao Fogo Cruzado, não um tiroteio: registro não é evento, e uma contagem de registros não é risco sem um denominador.

## Parte 1: os princípios em código

Diga que princípios cada gráfico quebra e redesenhe-o com título ativo.

Três gráficos do Fogo Cruzado quebram cada um pelo menos dois princípios: uma pizza dos motivos, uma barra poluída dos bairros, uma linha alta dos meses. Para cada um, escreva o que está errado e depois redesenhe: o gráfico certo para a pergunta, ordenado, eixos com rótulo, título ativo e nada que dê para subtrair. Extra: anote o ponto de que o gráfico fala.

Dos slides, os princípios em poucas linhas:

In [ ]:
victims = df.groupby("year")["n_victims"].sum()

In [ ]:
ax = victims.loc[2017:2024].plot.bar(rot=0)
ax.set(xlabel="", ylabel="vítimas")
ax.set_title("As vítimas caíram 36% de 2017 a 2024")
plt.show()

### Gráfico A

In [ ]:
counts = recent["main_reason"].value_counts()
plt.pie(counts, labels=counts.index,
        colors=sns.color_palette("hls", len(counts)))
plt.title("Figura 1")
plt.show()

**1.1** Que princípios o gráfico A quebra?

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# **1.1** Que princípios o gráfico A quebra?
#
# *Resposta:* pede ao leitor que compare dezesseis ângulos em vez de posições, então as fatias pequenas e seus rótulos se atropelam; as cores de arco-íris não querem dizer nada; e "Figura 1" não é um título ativo. A única coisa que ele mostra, que "Não identificado" é a maior fatia, merecia ser o título.

**A FAZER 1.2** Redesenhe o gráfico A: barras horizontais, ordenadas, com rótulos e título ativo.

In [ ]:
# counts = ____  # registros por motivo, do menor para o maior
# ax = ____  # barras horizontais: posição, não ângulo
# ax.set(xlabel="registros desde 2023", ylabel="")
# ax.set_title(____)  # o achado, não "Figura 1"
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
counts = recent["main_reason"].value_counts().sort_values()
ax = counts.plot.barh()
ax.set(xlabel="registros desde 2023", ylabel="")
ax.set_title("42% dos registros desde 2023 não informam motivo")
sns.despine()
# também vale: tirar "Não identificado" das barras e dizer isso no título
plt.show()

### Gráfico B

In [ ]:
top = recent["neighborhood"].value_counts().head(10).sort_index()
ax = top.plot.bar(hatch="//", edgecolor="black")
ax.grid(True, color="black")
ax.legend(["contagem"])
plt.show()

**1.3** Que princípios o gráfico B quebra?

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# **1.3** Que princípios o gráfico B quebra?
#
# *Resposta:* a hachura, a grade preta e a legenda de uma entrada são tinta que não carrega dado e dá para subtrair; a ordem alfabética esconde o ranking; nada diz o que é contado nem o que o gráfico mostra. E uma contagem de registros por bairro não é risco: comparado a quê?

**A FAZER 1.4** Redesenhe o gráfico B: ordenado pelo valor, sem nada que dê para subtrair, eixo com rótulo e título ativo.

In [ ]:
# top = ____  # os 10 nomes mais citados, ordenados pelo valor
# ax = top.plot.barh()
# ax.set(xlabel=____, ylabel="")
# ax.set_title(____)
# sns.despine()
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
top = recent["neighborhood"].value_counts().head(10).sort_values()
ax = top.rename(bairro).plot.barh()
ax.set(xlabel="registros desde 2023", ylabel="")
ax.set_title("Cascadura, Praça Seca e Vila Isabel lideram em registros")
sns.despine()
# uma contagem de registros, não um risco: por habitante é a aula de segunda
plt.show()

### Gráfico C

In [ ]:
monthly = df.resample("MS", on="date").size()
fig, ax = plt.subplots(figsize=(3, 6))
ax.plot(monthly.values)
plt.show()

**1.5** Que princípios o gráfico C quebra?

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# **1.5** Que princípios o gráfico C quebra?
#
# *Resposta:* alto e estreito, faz toda mudança parecer mais íngreme do que é; o eixo x é o número da linha, não a data; não há rótulos nem título. E o último ponto é novembro de 2025, que só tem 24 dias.

**A FAZER 1.6** Redesenhe o gráfico C: datas no eixo x, figura larga, rótulos e título ativo.

In [ ]:
# fig, ax = plt.subplots(figsize=____)  # largo, não alto
# ax.plot(monthly.index, monthly.values)
# ax.set(xlabel="", ylabel=____)
# ax.set_title(____)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly.values)
ax.set(xlabel="", ylabel="registros por mês")
ax.set_title("Os registros por mês caíram de 592 (ago/2018) para uns 120 em 2025")
plt.show()

**A FAZER 1.7** *Extra.* Anote o ponto de que o gráfico fala: o mês com mais registros.

In [ ]:
# peak = monthly.idxmax()
# ax.annotate(____, xy=(peak, monthly.max()),
#             xytext=(pd.Timestamp("2020-06-01"), 520),
#             arrowprops={"arrowstyle": "->"})
# fig
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
peak = monthly.idxmax()
ax.annotate("mês com mais registros: ago/2018", xy=(peak, monthly.max()),
            xytext=(pd.Timestamp("2020-06-01"), 520),
            arrowprops={"arrowstyle": "->"})
fig

## A caixa de ferramentas, dos slides

Uma célula por gráfico da seção da caixa de ferramentas, como nos slides. Rode, troque uma coluna e copie daqui na Parte 2. Não existe gráfico certo para todo caso: cada pergunta tem alguns bons. Primeiro as duas tabelas pequenas que vários desses gráficos usam, uma por semana e uma por nome de bairro (os outros usam os registros direto):

In [ ]:
# registros por semana; a primeira e a última semana são parciais
weekly = recent.resample("W", on="date").size()
weeks = weekly.iloc[1:-1].rename("registros").reset_index()
weeks["ano"] = weeks["date"].dt.year.astype(str)
# uma linha por nome de bairro do Fogo Cruzado
nb = recent.groupby("neighborhood").agg(
    registros=("id", "size"), vítimas=("n_victims", "sum"),
    mortos=("n_dead", "sum"), ação_policial=("police_action", "sum"),
    longitude=("longitude", "mean"),
    latitude=("latitude", "mean"))

### Caixa, violino, pontos: como se distribui?

In [ ]:
kw = dict(data=weeks, x="registros", y="ano")
fig, axes = plt.subplots(1, 3, sharey=True)
sns.boxplot(**kw, ax=axes[0])
sns.violinplot(**kw, ax=axes[1])
sns.stripplot(**kw, ax=axes[2])
plt.show()

### Extra: violino com os pontos

In [ ]:
sns.violinplot(**kw, inner=None, color="0.85")
sns.stripplot(**kw, color="black", size=3)
plt.show()

### Barras de erro: quanta certeza tem a média?

In [ ]:
sns.pointplot(weeks, x="ano", y="registros",
              errorbar=("ci", 95), linestyle="none")
plt.show()

### Gráfico de contagem: quantos em cada grupo?

In [ ]:
listed = {True: "com vítimas", False: "sem vítimas"}
df["vítimas"] = (df["n_victims"] > 0).map(listed)
ax = sns.countplot(df, x="year", hue="vítimas")
ax.set(xlabel="", ylabel="registros")
plt.show()

### Barras agrupadas: dois números dentro de cada grupo

In [ ]:
cols = ["n_dead", "n_wounded"]
dead_wounded = recent.groupby("year")[cols].sum()
dead_wounded.columns = ["mortos", "feridos"]
ax = dead_wounded.plot.bar(rot=0)
ax.set(xlabel="", ylabel="vítimas")
plt.show()

### Barras empilhadas e 100% empilhadas: que parte do todo?

In [ ]:
counts = pd.crosstab(df["year"], df["police_action"])
counts = counts.rename(columns={False: "sem ação policial",
                                True: "com ação policial"})
counts = counts.rename_axis(index="ano", columns=None)
counts.plot.bar(stacked=True)
shares = counts.div(counts.sum(axis=1), axis=0)
shares.plot.bar(stacked=True)
plt.show()

### Rosca: a parte de duas ou três partes

In [ ]:
known = recent["main_reason"] != "Não identificado"
names = {True: "com motivo", False: "sem motivo"}
parts = known.map(names).value_counts()
plt.pie(parts, labels=parts.index, autopct="%.0f%%",
        wedgeprops={"width": 0.4}, pctdistance=0.8)
plt.show()

### Área empilhada: um total no tempo, e suas partes

In [ ]:
people = df.groupby("year")[["n_civilians", "n_agents"]].sum()
people.columns = ["civis", "agentes"]
ax = people.plot.area()
ax.set(xlabel="", ylabel="vítimas")
plt.show()

### Bolhas: um terceiro número como tamanho

In [ ]:
sns.scatterplot(nb, x="longitude", y="latitude",
                size="registros", sizes=(5, 400))
plt.axis("equal")
plt.show()

### Matriz de dispersão: todos os pares de colunas de uma vez

In [ ]:
g = sns.pairplot(nb, vars=["registros", "vítimas", "ação_policial"])
for ax in g.diag_axes:
    ax.set_ylabel("")
plt.show()

### Mapa de calor de correlação: que colunas andam juntas?

In [ ]:
cols = ["registros", "vítimas", "mortos", "ação_policial"]
corr = nb[cols].corr()
sns.heatmap(corr, annot=corr.map(lambda x: br(x, 2)), fmt="",
            cmap="vlag", vmin=-1, vmax=1)
plt.show()

### Pequenos múltiplos: um painel por grupo

Primeiro a tabela: registros por trimestre dos quatro nomes de bairro mais citados, só trimestres completos.

In [ ]:
top4 = recent["neighborhood"].value_counts().index[:4]
four = recent[recent["neighborhood"].isin(top4)]
four = four[four["date"] < "2025-10-01"]  # o último trimestre acaba em 24/11
quarter = pd.Grouper(key="date", freq="QS")
by_q = four.groupby(["neighborhood", quarter]).size()
quarterly = by_q.reset_index(name="registros")
quarterly["bairro"] = quarterly["neighborhood"].map(bairro)

In [ ]:
g = sns.relplot(quarterly, x="date", y="registros",
                col="bairro", kind="line")
g.set_titles("{col_name}")
g.set_axis_labels("", "registros por trimestre")
plt.show()

### Extra: matriz de confusão, onde o modelo erra?

O Fogo Cruzado não tem classificador, então este exemplo usa a tabela iris que vem com o scikit-learn: uma regressão logística que separa três espécies de íris por duas medidas, escolhida porque erra. Os nomes das espécies e das medidas ficam em inglês, como na tabela.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
X = iris.data[["sepal length (cm)", "sepal width (cm)"]]
y = iris.target_names[iris.target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0)
model = LogisticRegression(max_iter=1000)
y_pred = model.fit(X_train, y_train).predict(X_test)
labels = model.classes_

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize="true")
ax = sns.heatmap(cm, annot=np.vectorize(lambda x: br(x, 2))(cm), fmt="",
                 cmap="Greens", xticklabels=labels, yticklabels=labels)
ax.set(xlabel="previsto", ylabel="real")
plt.show()

## Parte 2: escolha o gráfico

Responda às cinco perguntas, cada uma com um gráfico e um título ativo.

Cinco perguntas, um gráfico cada: registros por dia em cada dia da semana; quanta certeza tem a média de vítimas por registro, ano a ano; a fatia dos registros de cada ano que informam um motivo; registros contra vítimas, por bairro; registros por ano para os motivos mais comuns, um painel cada. Títulos ativos. Mais de um gráfico pode estar certo: esteja pronto para dizer por que escolheu o seu.

In [ ]:
# Os nomes dos dias em português, na ordem da semana (o pandas devolve em inglês).
DIAS = {"Monday": "segunda", "Tuesday": "terça", "Wednesday": "quarta",
        "Thursday": "quinta", "Friday": "sexta", "Saturday": "sábado",
        "Sunday": "domingo"}

**A FAZER 2.1** Como se distribuem os registros por dia em cada dia da semana? (`recent`: `date`)

In [ ]:
# daily = recent.resample("D", on="date").size()
# days = daily.rename("registros").reset_index()
# days["dia"] = days["date"].dt.day_name().map(DIAS)
# ax = ____  # uma distribuição por dia da semana
# ax.set_title(____)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
daily = recent.resample("D", on="date").size()
days = daily.rename("registros").reset_index()
days["dia"] = days["date"].dt.day_name().map(DIAS)
order = list(DIAS.values())
ax = sns.boxplot(days, x="registros", y="dia", order=order)
ax.set_title("Sábado e domingo têm menos registros, mas as faixas se sobrepõem")
# também vale: sns.violinplot ou sns.stripplot com o mesmo x e y
plt.show()

**A FAZER 2.2** Quanta certeza tem a média de vítimas por registro, ano a ano, nos registros com pelo menos uma vítima? (`recent`: `n_victims`, `year`)

In [ ]:
# hit = recent[recent["n_victims"] > 0]
# ax = ____  # a média, com o intervalo de 95%
# ax.set_title(____)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
hit = recent[recent["n_victims"] > 0]
ax = sns.pointplot(hit, x="year", y="n_victims",
                   errorbar=("ci", 95), linestyle="none")
ax.yaxis.set_major_formatter(BR)
ax.set(xlabel="", ylabel="vítimas por registro (média)")
ax.set_title("O intervalo de 2025 é largo: um registro tem 145 vítimas")
plt.show()

**A FAZER 2.3** Que fatia dos registros de cada ano informa um motivo? (`df`: `main_reason`, `year`)

In [ ]:
# given = df["main_reason"] != "Não identificado"
# shares = ____  # uma linha por ano, as fatias somando 1
# ax = ____
# ax.set_title(____)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
given = df["main_reason"] != "Não identificado"
shares = pd.crosstab(df["year"], given, normalize="index")
ax = shares.plot.bar(stacked=True, rot=0)
ax.legend(["sem motivo", "com motivo"], loc="center left",
          bbox_to_anchor=(1, 0.5))
ax.set(xlabel="", ylabel="fatia dos registros")
ax.set_title("Registros com motivo: 64% em 2025, o maior da série")
plt.show()

**A FAZER 2.4** Por bairro, mais registros vêm com mais vítimas? Qual bairro quebra o padrão? (`nb`: `registros`, `vítimas`)

In [ ]:
# ax = ____  # um ponto por bairro
# ax.set_title(____)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
ax = sns.scatterplot(nb, x="registros", y="vítimas")
ax.annotate("Penha: 145 vítimas num só registro", xy=(91, 221),
            xytext=(110, 200), arrowprops={"arrowstyle": "->"})
ax.set_title("Mais registros, mais vítimas, exceto na Penha")
plt.show()

**A FAZER 2.5** Como mudaram os registros por ano dos quatro motivos mais comuns, um painel cada? Use os anos completos de 2017 a 2024. (`df`: `main_reason`, `year`)

In [ ]:
# main4 = df["main_reason"].value_counts().index[:4]
# full = df[df["main_reason"].isin(main4) & df["year"].between(2017, 2024)]
# per_year = ____  # registros por motivo e ano, em formato longo
# g = ____  # um painel por motivo, mesmos eixos
# g.figure.suptitle(____, y=1.03)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
main4 = df["main_reason"].value_counts().index[:4]
full = df[df["main_reason"].isin(main4) & df["year"].between(2017, 2024)]
per_year = full.groupby(["main_reason", "year"]).size()
per_year = per_year.reset_index(name="registros")
g = sns.relplot(per_year, x="year", y="registros", col="main_reason",
                kind="line", col_wrap=2)
g.set_titles("{col_name}")
g.set_xlabels("")
g.figure.suptitle("Quase toda a queda desde 2018 está nos registros sem motivo",
                  y=1.03)
plt.show()

**A FAZER 2.6** *Extra.* Faça a 2.2 de novo sem o registro de 28/10/2025. O que muda?

In [ ]:
# record = (hit["date"] == "2025-10-28") & (hit["n_victims"] == 145)
# ax = ____  # o mesmo gráfico, com hit[~record]
# ax.set_title(____)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
record = (hit["date"] == "2025-10-28") & (hit["n_victims"] == 145)
ax = sns.pointplot(hit[~record], x="year", y="n_victims",
                   errorbar=("ci", 95), linestyle="none")
ax.yaxis.set_major_formatter(BR)
ax.set(xlabel="", ylabel="vítimas por registro (média)")
ax.set_title("Sem um registro, 2025 fica igual a 2024")
plt.show()

**2.7** Escolha uma pergunta em que outro gráfico também teria servido. Qual, e por que você escolheu o seu?

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# **2.7** Uma pergunta em que outro gráfico também teria servido.
#
# *Uma resposta:* a 2.1. A caixa resume cada dia da semana; o violino mostra que a maioria dos dias fica numa faixa estreita; os pontos mostram, um por um, os poucos dias com mais de 15 registros. Os três respondem à pergunta.

## Como gráficos mentem: as duas revisões, em código

Os eixos honestos de terça: uma linha decide onde começa o eixo de uma barra. Primeiro a média semanal por ano:

In [ ]:
weekly_mean = weeks.groupby("ano")["registros"].mean()

In [ ]:
fig, (corte, zero) = plt.subplots(1, 2)
weekly_mean.plot.bar(ax=corte, rot=0).set_ylim(28, 39)
weekly_mean.plot.bar(ax=zero, rot=0)  # honesto: do zero
plt.show()

Dois eixos y podem ser escalados para fazer quaisquer duas linhas se cruzarem. Ponha as duas séries num eixo só, com o mesmo início:

In [ ]:
full = df[df["year"].between(2017, 2024)].groupby("year")
yearly = pd.DataFrame({"registros": full.size(),
                       "vítimas": full["n_victims"].sum()})
yearly.index.name = "ano"

In [ ]:
indexed = yearly / yearly.iloc[0] * 100
ax = indexed.plot(ylabel="2017 = 100")
plt.show()

## Parte 3: carros, contagem e taxa

Cinco passos, da manchete ao título ativo. Os passos seguem o bloco dos slides: a tabela dos mais furtados, a manchete, a taxa, o corte de frota e o título ativo.

**Os dados.** Estado de São Paulo, porque o Rio não publica furtos e roubos por modelo. São 120 modelos de carro com pelo menos um furto ou roubo em 2025. `thefts`: veículos furtados ou roubados em 2025, um por veículo, só carros e utilitários leves, recuperados ainda contam (SSP-SP, *Veículos subtraídos 2025*, um arquivo bruto que "não constitui estatística oficial"). `fleet`: veículos registrados no estado em junho de 2025, de todos os anos de fabricação (Senatran/Renavam, domínio público). Versões do mesmo modelo foram somadas (HB20 com HB20S, Onix com Onix Plus). O ranking dos slides usa só os 79 modelos com 50.000 ou mais registrados. Os nomes das colunas ficam em inglês: `thefts` (furtos e roubos), `fleet` (frota), `furto`, `roubo`.

In [ ]:
# Os carros dos slides: um modelo por linha, só contagens agregadas.
# Furtos e roubos: SSP-SP, Veículos subtraídos 2025 (ssp.sp.gov.br, transparência).
# Frota: Senatran/Renavam, frota por UF, município, marca e modelo, junho
# de 2025 (dados.transportes.gov.br). Preparada para a aula em 06/11/2026.
import io

CARS_CSV = """\
model,thefts,furto,roubo,fleet
Hyundai HB20,3109,2626,483,552990
Volkswagen Gol,2795,2659,136,2094113
Chevrolet Onix,2519,2247,272,569756
Ford Ka,2263,2119,144,486412
Fiat Uno,1976,1893,83,1067733
Chevrolet Corsa,1784,1718,66,822910
Fiat Argo,1512,1363,149,111049
Fiat Strada,1383,1305,78,372726
Fiat Palio,1371,1282,89,1126441
Fiat Mobi,1354,1246,108,121182
Volkswagen Fox,1131,1052,79,488562
Renault Kwid,967,835,132,141100
Jeep Renegade,914,804,110,187758
Ford Fiesta,884,785,99,667635
Jeep Compass,864,798,66,138311
Volkswagen Saveiro,808,747,61,382748
Chevrolet Prisma,800,736,64,251661
Nissan Kicks,799,703,96,158057
Renault Sandero,791,708,83,259152
Volkswagen Voyage,711,641,70,340301
Toyota Hilux,690,653,37,118955
Honda Fit,676,588,88,309954
Fiat Toro,660,571,89,113427
Ford EcoSport,576,505,71,244881
Honda HR-V,551,476,75,175752
Chevrolet Spin,536,484,52,112865
Fiat Siena,529,499,30,270701
Fiat Cronos,506,466,40,36582
Honda Civic,499,439,60,281836
Chevrolet Celta,456,404,52,542436
Renault Logan,454,408,46,118286
Toyota Corolla,454,358,96,320284
Volkswagen Polo,453,295,158,289467
Chevrolet Cobalt,377,347,30,84065
Volkswagen T-CROSS,368,276,92,158695
Hyundai Tucson,349,322,27,87398
Citroen C3,333,289,44,172522
Hyundai I30,330,310,20,54601
Hyundai Creta,317,217,100,154001
Honda City,293,240,53,164429
Nissan Frontier,260,247,13,34535
Fiat Idea,255,236,19,97876
Chevrolet Kadett,242,242,0,175490
Chevrolet Montana,234,204,30,189578
Chevrolet S10,233,206,27,174143
Chevrolet Tracker,231,155,76,127589
Nissan Sentra,222,214,8,34948
Volkswagen Parati,217,214,3,275425
Nissan Versa,214,176,38,74730
Chevrolet Classic,212,189,23,314443
Volkswagen Kombi,212,198,14,384322
Volkswagen Amarok,199,162,37,42918
Chevrolet Cruze,197,170,27,87238
Volkswagen Up,192,169,23,81374
Volkswagen Nivus,189,122,67,89119
Fiat Pulse,187,175,12,36805
Volkswagen Jetta,187,145,42,55243
Toyota Etios,177,137,40,127510
Toyota Yaris,177,138,39,113347
Renault Duster,172,146,26,94843
Volkswagen Golf,157,126,31,127056
Fiat Punto,154,137,17,79445
Ford Focus,141,119,22,88660
Ford Ranger,138,106,32,93618
Ford Escort,136,130,6,381433
Chevrolet Vectra,135,118,17,202375
Peugeot 208,130,104,26,52156
Chevrolet Meriva,126,111,15,105629
Fiat Fastback,125,107,18,37359
Peugeot 207,124,114,10,81129
Renault Clio,121,108,13,149285
Chevrolet Astra,117,96,21,208524
Nissan March,110,93,17,66050
Volkswagen Virtus,109,84,25,61555
Peugeot 206,107,96,11,130526
Volkswagen Tiguan,107,84,23,31081
Volkswagen Spacefox,102,90,12,68082
Chevrolet Agile,97,84,13,90267
Volkswagen Santana,97,95,2,196700
Citroen C4,95,85,10,55515
Hyundai IX35,95,70,25,44043
Toyota Corolla Cross,92,73,19,62071
Volkswagen Fusca,84,79,5,680647
Citroen C3 AIRCROSS,74,61,13,29627
Renault Oroch,64,53,11,20457
Volkswagen Crossfox,64,51,13,52092
Toyota SW4,62,56,6,20647
Ford Fusion,61,51,10,29612
Honda CR-V,60,47,13,39308
Renault Captur,56,39,17,30841
Chevrolet Zafira,50,42,8,57420
Honda WR-V,44,37,7,23336
Jeep Commander,44,35,9,20208
Volkswagen Passat,43,38,5,187173
Peugeot 2008,40,34,6,23204
Peugeot 307,40,36,4,33120
Fiat Linea,39,33,6,17516
Fiat Stilo,37,32,5,35685
Citroen C4 CACTUS,36,22,14,24205
Fiat Freemont,33,28,5,7955
Chevrolet Sonic,30,27,3,10503
Fiat 500,30,28,2,14520
Fiat Bravo,29,28,1,11287
Renault Fluence,25,14,11,14106
Toyota RAV4,24,18,6,16823
Ford Edge,20,16,4,5303
Renault Megane,19,14,5,23862
Volkswagen Taos,16,11,5,24490
Renault Kardian,12,6,6,9596
Honda Accord,9,6,3,8986
Fiat Marea,7,7,0,28638
Volkswagen Bora,6,6,0,6983
Chevrolet Equinox,5,3,2,7176
Fiat Tempra,5,5,0,89527
Byd Dolphin,4,3,1,14624
Ford Territory,4,4,0,3030
Jeep Cherokee,2,2,0,3550
Ford Bronco,1,1,0,2338
Ford Maverick,1,0,1,8561
Toyota Camry,1,1,0,3930
"""


def cars_file():
    return io.StringIO(CARS_CSV)  # um arquivo novo a cada leitura

**A FAZER 3.1** Monte a tabela dos sete modelos mais furtados e roubados.

In [ ]:
# cars_all = pd.read_csv(cars_file(), index_col="model")  # os 120 modelos
# cars = cars_all[cars_all["fleet"] >= 50_000].copy()  # os 79 do ranking
# top = ____  # os sete com mais furtos e roubos
# top[["thefts"]].style.format(br)  # números com ponto de milhar
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
cars_all = pd.read_csv(cars_file(), index_col="model")
cars = cars_all[cars_all["fleet"] >= 50_000].copy()
top = cars.nlargest(7, "thefts")
top[["thefts"]].style.format(br)

**3.2** Escreva a manchete que um jornal daria para essa tabela, sobre o HB20.

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# **3.2** A manchete do HB20.
#
# *Uma resposta:* "O HB20 é o carro mais furtado e roubado de SP". É verdade (3.109 em 2025) e convida quem lê a concluir que ter um HB20 é perigosíssimo. Comparado a quê? A tabela ainda não diz quantos HB20 existem: são 552.990 registrados, contra 2.094.113 Gol e 111.049 Argo.

**A FAZER 3.3** Calcule a taxa por 1.000 registrados e a posição de cada modelo nas duas ordens, contagem e taxa.

In [ ]:
# cars["per_1000"] = ____  # furtos e roubos por 1.000 veículos registrados
# ranks = ____  # a posição na contagem e na taxa, 1 = o maior
# ranks.loc[["Hyundai HB20", "Volkswagen Gol", "Fiat Argo"]]
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
# O HB20 não ficou seguro: 5,62 por 1.000 é três vezes a mediana dos 79 (1,81).
# A contagem ignora o tamanho da frota: o Gol tem 3,8 vezes a frota do HB20 e
# menos furtos e roubos; comparada à frota, a ordem muda.
cars["per_1000"] = 1000 * cars["thefts"] / cars["fleet"]
top = cars.nlargest(7, "thefts")
print("mediana dos 79:", br(cars["per_1000"].median(), 2))
top[["thefts", "fleet", "per_1000"]].style.format(
    {"thefts": br, "fleet": br, "per_1000": lambda x: br(x, 2)})

As posições nas duas ordens, num gráfico de halteres, como no slide (uma seta da posição na contagem à posição por 1.000): o HB20 cai de 1º para 8º, o Gol de 2º para 58º, o Argo sobe de 7º para 1º.

In [ ]:
ranks = cars[["thefts", "per_1000"]].rank(ascending=False)
pick = ["Hyundai HB20", "Volkswagen Gol", "Fiat Uno",
        "Fiat Palio", "Fiat Argo"]
for i, (n, r) in enumerate(ranks.loc[pick].values):
    plt.annotate("", xy=(r, i), xytext=(n, i),
                 arrowprops={"arrowstyle": "->"})  # da contagem à taxa
plt.yticks(range(len(pick)), pick)
plt.xlabel("posição: da contagem (início) à taxa (seta)")
plt.show()

**A FAZER 3.4** Descubra quantos modelos entram no ranking, e quem fica no topo da taxa, se o corte de frota for 5, 1.000 ou 50.000 registrados.

In [ ]:
# for cut in (5, 1_000, 50_000):
#     kept = ____  # os modelos com pelo menos `cut` registrados
#     per_1000 = ____  # a taxa só desses modelos
#     print(cut, len(kept), per_1000.idxmax(), round(per_1000.max(), 2))
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
for cut in (5, 1_000, 50_000):
    kept = cars_all[cars_all["fleet"] >= cut]
    per_1000 = 1000 * kept["thefts"] / kept["fleet"]
    print(f"corte {br(cut)}: {len(kept)} modelos; no topo,",
          per_1000.idxmax(), br(per_1000.max(), 2))

# Com 5 ou 1.000, entram os 120 modelos (o menor tem 2.338 registrados) e o
# Cronos, com 36.582 registrados, passa o Argo: 13,83 contra 13,62 por 1.000.
# Com 50.000 ficam 79. Numa frota pequena, cada furto a mais mexe muito na taxa:
# com 5 carros, um roubo já é 20%. O corte é uma escolha; diga qual usou.

**A FAZER 3.5** Desenhe as dez maiores taxas e dê ao gráfico um título ativo.

In [ ]:
# rate = ____  # as dez maiores taxas, da menor para a maior
# ax = rate.plot.barh()
# ax.xaxis.set_major_formatter(BR)
# ax.set(xlabel="furtados ou roubados por 1.000 registrados", ylabel="")
# ax.set_title(____)  # o achado, não "Taxa por modelo"
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
rate = cars["per_1000"].nlargest(10).sort_values()
ax = rate.plot.barh()
ax.xaxis.set_major_formatter(BR)
ax.set(xlabel="furtados ou roubados por 1.000 registrados", ylabel="")
ax.set_title("Por carro registrado, o Argo lidera")
plt.show()

**3.6** *Extra.* Argo, Mobi e Toro são modelos novos. Por que só isso já pode empurrar a taxa deles para cima?

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# **3.6** Modelos novos.
#
# *Resposta:* a frota conta todo carro registrado do modelo, de todos os anos. Modelos antigos (Gol, Palio, Uno) têm milhões de carros velhos no denominador, e quem furta pode preferir certas idades; um modelo novo só tem carros recentes. A taxa é por carro registrado do modelo, não por carro da mesma idade, então não há ajuste por idade: comparado a quê?, um nível abaixo.

## Parte 4: conserte este gráfico

Liste os problemas e redesenhe: em que bairros os registros listam mais vítimas?

A célula abaixo desenha o gráfico do slide. Escreva todos os problemas que achar e depois redesenhe o gráfico para responder com honestidade: "em que bairros os registros listam mais vítimas desde 2023?". Um gráfico por dupla; no fim olhamos três.

In [ ]:
by_nb = recent.groupby("neighborhood")["n_victims"]
top5 = by_nb.sum().nlargest(5).index
hit = recent[recent["n_victims"] > 0]
per_report = hit.groupby("neighborhood")["n_victims"].mean()
bad = per_report[top5].sort_index()
ax = bad.plot.bar(rot=0, color=list("rgbmy"))
ax.set_ylim(1.2, 8)
ax.set_title("Figura 1")
plt.show()

**4.1** Escreva todos os problemas que achar.

*Sua resposta:*

In [ ]:
# @title Solução { display-mode: "form" }
# **4.1** Todos os problemas.
#
# *Resposta:* pelo menos cinco.
#
# 1. O eixo começa em 1,2, então a barra de Bangu quase some e a Penha parece umas trinta vezes maior, quando a média dela é 5,4 vezes a de Bangu.
# 2. Mostra a média de uma variável assimétrica: um registro com 145 vítimas leva a média da Penha a 7,6; a mediana é 1.
# 3. Cinco cores de arco-íris que não querem dizer nada.
# 4. A ordem alfabética esconde o ranking.
# 5. "Figura 1", nenhum rótulo no eixo y e o nome cru da coluna `neighborhood` no eixo x.
# 6. Responde a "vítimas por registro", não à pergunta feita; e nem um total nem uma média por registro é risco para quem mora lá: comparado a quê?

**A FAZER 4.2** Redesenhe para responder com honestidade: "em que bairros os registros listam mais vítimas desde 2023?"

In [ ]:
# total = ____  # vítimas por bairro, os cinco maiores, ordenados
# ax = ____  # barras horizontais desde o zero
# ax.set(xlabel=____, ylabel="")
# ax.set_title(____)  # um título ativo; diga qual é o registro recorde
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
total = by_nb.sum().nlargest(5).sort_values()
ax = total.rename(bairro).plot.barh()
ax.set(xlabel="mortos ou feridos, 2023-2025", ylabel="")
ax.set_title("Penha: 221 vítimas, 145 num só registro")
plt.show()

**A FAZER 4.3** *Extra.* Faça a mesma pergunta por registro em vez de no total: a fatia dos registros de cada bairro que lista uma vítima. O que acontece com a ordem?

In [ ]:
# reports = ____  # registros por bairro
# with_v = ____  # registros com pelo menos uma vítima
# share = ____  # na ordem da 4.2, em %
# ax = share.plot.barh()
# ax.set(xlabel="% dos registros com vítima", ylabel="")
# ax.set_title(____)
# seu código aqui

In [ ]:
# @title Solução { display-mode: "form" }
reports = recent.groupby("neighborhood").size()
with_v = hit.groupby("neighborhood").size()
share = (with_v / reports)[total.index] * 100
ax = share.rename(bairro).plot.barh()
ax.set(xlabel="% dos registros com vítima", ylabel="")
ax.set_title("Por registro, Bangu lidera: 43%")
plt.show()

Os mesmos dados respondem a duas perguntas, em ordens diferentes: no total, a Penha vem primeiro, por causa de um registro; por registro, Bangu. Qual gráfico é honesto depende da pergunta, então diga a que pergunta o seu gráfico responde, e lembre que nenhum dos dois é risco para quem mora lá.